# EV Charging Demand & Power Prediction

This notebook builds and honestly evaluates the single model EVOptima ships, and
documents why the *second* model it used to ship was measured and removed.

| model | algorithm | target | what the view does with it |
|---|---|---|---|
| `power` | `XGBRegressor` | `Charging Power_kW` | charging power (kW) |
| -- | -- | -- | `rate = power`, `energy = power x duration` (arithmetic) |

**Headline results** (row-weighted blocked CV, run *within* each source):

| model | blocked CV R² | chronological 80/20 | latency | size |
|---|---|---|---|---|
| `power` | **+0.9960** | +0.9960 / +0.9984 | 0.34 ms/row | 3.20 MB |

**Finding 1b** (sections 1.1-1.3) is the reason there is only one row. A second
model for the average charging rate was trained, shipped and scored **+0.9977** --
and the score was algebraically circular: `Charging Rate_kW x Charging_Duration_h`
is *identically* `Energy Supplied_kWh` in this data, so the metric reduced to
predicting a near-constant, where a **constant fits better (0.9996)**. In service
it was **75 % wrong** (251 % worst case), so it was removed.

The point of this notebook is **not** the number -- it is that the number is
reachable *without cheating*. The previous version reported R² ≈ 1.00 from a
random `train_test_split` over a cumulative target. Section 1 finds that leak,
measures it, and removes it; sections 2-6 rebuild the evaluation so the figure
survives contact with unseen data.

Everything here imports from `ml/pipeline/`, the same package the Django view
serves from, so the notebook and production cannot drift apart.


## What was wrong with the previous version

1. **Random split on a 5-second time series.** `train_test_split(random_state=42)`
   puts row *t+1* in training while scoring row *t*. Consecutive rows are near
   duplicates, so the score is inflated (0.95 on a time split becomes 1.00).
2. **The target is a cumulative ramp.** `Energy Supplied_kWh` only ever grows
   with elapsed time -- `corr(Energy, Duration) = +1.0000`. A tree fitted to it
   learns *how long the session has been running*, not charging behaviour, and
   cannot extrapolate past the durations it saw. On `ev_charging_data1.csv` it
   scores **R² = -3.16** under a chronological split and **-3.26** under
   blocked CV.
3. **Leaked features.** `Time Elapsed_s` and `Charging_Duration_h` are
   components of the target rather than causes of it; inside
   `ev_charging_data1.csv`, `State Of Charge_SoC` also correlates +1.0 with it.
4. **Calendar features acting as dataset identifiers.** `month` ∈ {2, 3} and
   `day_of_week` ∈ {1, 2, 3, 4} separate the two sources perfectly, so the model
   answers from the *date* instead of from the operator's inputs -- a 12 kWh
   session came back as 54 kWh.
5. **`is_weekend` was constant** (always 0): a zero-variance feature.
6. **It wrote to the repo root** (`model.joblib`, `scaler.joblib`) with no
   version, no provenance and no recorded feature order.
7. **A second model was shipped for a quantity that is arithmetic.** Sections 1.1-1.3
   measure it and remove it.

Sections 1-6 are the replacement.


In [1]:
import json
import os
import pathlib
import shutil
import sys
import time

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Repo root, so this works whether Jupyter was launched from the repository
# root or from ml/notebooks/.
ROOT = pathlib.Path.cwd().resolve()
if not (ROOT / "ml").is_dir():
    ROOT = ROOT.parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Only the export step needs Django (settings.MODEL_DIR); the analysis does not.
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "config.settings.dev")
import django

django.setup()

from django.conf import settings

from ml.pipeline import audit as audit_mod
from ml.pipeline import config, data, evaluate, train
from ml.pipeline.config import (
    ENERGY_FEATURES,
    POWER_FEATURES,
    POWER_MODEL_NAME,
    PROTOCOL,
    SPECS,
)

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 50)

print(f"repo root   : {ROOT}")
print(f"artifacts   : {settings.MODEL_DIR}")
print(f"protocol    : {PROTOCOL}")
print(f"derived     : {config.ENERGY_DERIVED}")


repo root   : D:\projets\EVOptima
artifacts   : D:\projets\EVOptima\ml\artifacts
protocol    : chronological 80/20 holdout plus TimeSeriesSplit(5) blocked CV, run independently within each source so no fold spans the d1 -> d2 operating-regime boundary
derived     : energy_kwh = charging_power_kw * duration_h


In [2]:
datasets = data.load_datasets()

overview = pd.DataFrame(
    {
        source: {
            "rows": len(frame),
            "start": frame[data.TIMESTAMP_COL].min(),
            "end": frame[data.TIMESTAMP_COL].max(),
            "span_h": round(
                (
                    frame[data.TIMESTAMP_COL].max() - frame[data.TIMESTAMP_COL].min()
                ).total_seconds()
                / 3600,
                2,
            ),
            "sample_s": int(frame[data.TIMESTAMP_COL].diff().dt.total_seconds().median()),
            "power_kW": f"{frame['Charging Power_kW'].min():.1f}-{frame['Charging Power_kW'].max():.1f}",
            "voltage_V": f"{frame['Charging Voltage_V'].min():.0f}-{frame['Charging Voltage_V'].max():.0f}",
            "SoC_%": f"{frame['State Of Charge_SoC'].min():.1f}-{frame['State Of Charge_SoC'].max():.1f}",
            "hour": f"{frame['hour'].min()}-{frame['hour'].max()}",
            "month": sorted(frame["month"].unique().tolist()),
        }
        for source, frame in datasets.items()
    }
)

print("The two sources are NOT exchangeable -- different regimes, a month apart:")
print("fitting one CV across both puts a fold over that boundary and scores 0.72")
print("on a model that scores 0.996 everywhere else.\n")
overview.T

The two sources are NOT exchangeable -- different regimes, a month apart:
fitting one CV across both puts a fold over that boundary and scores 0.72
on a model that scores 0.996 everywhere else.



,rows,start,end,span_h,sample_s,power_kW,voltage_V,SoC_%,hour,month
d1,3006,2025-02-20 03:26:00,2025-02-20 07:36:00,4.17,0,1.0-7.0,200-500,20.0-53.1,3-7,[2]
d2,9087,2025-03-25 00:00:00,2025-03-28 06:18:00,78.3,0,5.0-50.0,300-800,20.0-24.2,0-6,[3]


# 1. Leakage audit

Before choosing a model, establish what the target actually *is* and which
columns are carrying the answer rather than a cause. Every number in this
section is produced by `ml/pipeline/audit.py` under the same blocked-CV
protocol, so the choice stops being a matter of opinion.

## 1.1 Is `Energy Supplied_kWh` just duration in disguise?

Three measurements, all produced by `ml/pipeline/evaluate.py`:

* `corr(Energy, Duration)` ≈ **+1** -- duration alone explains it;
* `corr(Energy, Power)` ≈ **0** -- the *instantaneous* power column explains
  nothing;
* `R²(Energy = mean_power x Duration)` ≈ **1** -- a constant does as well as
  any model.

So a tree trained on it learns *how long the session has been running*.

**The apparent paradox.** Real energy *is* `power x time`, so how can the column
be uncorrelated with power? It cannot be synthesised -- and it is not. The column
*is* exactly `Charging Rate_kW x Charging_Duration_h` (0.00000000 kWh), and it
reproduces `integral P dt` to **0.005 kWh (d1)** / **0.098 kWh (d2)** per row. It
decorrelates from *instantaneous* power for the same reason any running total
does: within a source the power series is stationary, so
the running mean converges to `k` almost immediately and `Energy ≈ k x duration`.
That is a fact about the **data**, not about the estimator -- and it is what
removes the second model. See Finding 1b.


In [3]:
for source, frame in datasets.items():
    d = evaluate.target_is_a_ramp(frame)
    print(
        f"{source}:  corr(E, Duration) = {d['corr_energy_duration']:+.4f}   "
        f"corr(E, Power) = {d['corr_energy_power']:+.4f}   "
        f"R2(E = mean_power x Duration) = {d['constant_k_r2']:.4f}"
    )

d1:  corr(E, Duration) = +1.0000   corr(E, Power) = -0.0107   R2(E = mean_power x Duration) = 0.9999
d2:  corr(E, Duration) = +0.9997   corr(E, Power) = +0.0098   R2(E = mean_power x Duration) = 0.9994


A ramp is a **monotone function of elapsed time**, so a tree model trained on it
can only reproduce durations it has already seen: it interpolates perfectly and
extrapolates to nonsense. That is exactly what happens under a chronological
split, where the test window contains durations longer than anything in the
training window.

The obvious fix is a better *target*: divide by duration and predict the average
**charging rate** (kW). That formulation is evaluated in 1.2 -- and scores
**+0.9977** -- and then **rejected in Finding 1b**, because the rate is
unlearnable too (CV 2.6-3.2 % against a power column that swings 6-10x *inside
the same source*).

What survives is not a better model but **no model at all** for kWh:
`energy = power x duration`, which is exact.


## 1.2 Three formulations, one protocol

Same data, same estimator, same blocked-CV -- only the framing changes.

| formulation | what it predicts | result |
|---|---|---|
| `naive + duration` | cumulative kWh, duration as a feature | −0.095 |
| `naive - duration` | cumulative kWh, duration removed | −0.330 |
| **`rate -> integrate`** | **rate, then integrate** *(shipped, later retired)* | **+0.9977** |

The +0.9977 is real, and it is still the number that was published. It is also
**the wrong number to trust**: it is scored against the *cumulative* ramp, where
a **constant** scores **0.9999 (d1) / 0.9994 (d2)** -- better than the model.
The next two cells take that apart.


In [4]:
naive_with_duration = ENERGY_FEATURES + [audit_mod.DURATION_COLUMN]

# SPECS now describes one model; the retired energy formulation is still
# evaluated here because it is the evidence behind Finding 1b.
estimator = config.energy_estimator

formulations = {
    "naive + duration": evaluate.per_source_report(
        datasets,
        naive_with_duration,
        config.ENERGY_TARGET,
        estimator,
    ),
    "naive - duration": evaluate.per_source_report(
        datasets,
        ENERGY_FEATURES,
        config.ENERGY_TARGET,
        estimator,
    ),
    "rate -> integrate (retired)": evaluate.per_source_report(
        datasets,
        ENERGY_FEATURES,
        config.RATE_TARGET,
        estimator,
        metric_target=config.ENERGY_TARGET,
        integrate=evaluate.integrate_energy,
    ),
}

rows = {
    label: {
        "blocked CV R2": rep["weighted_blocked_cv_r2"],
        **{f"chrono {s}": v["r2"] for s, v in rep["chronological_holdout"].items()},
        **{f"blocked {s}": v["r2"] for s, v in rep["blocked_cv"].items()},
    }
    for label, rep in formulations.items()
}
pd.DataFrame(rows).T.round(4)


,blocked CV R2,chrono d1,chrono d2,blocked d1,blocked d2
naive + duration,-0.0951,-3.1570,0.9958,-3.2600,0.9518
naive - duration,-0.3302,-3.1570,0.3779,-3.2600,0.6390
rate -> integrate (retired),0.9977,0.9968,0.9955,0.9986,0.9975


## 1.3 Calendar features are dataset identifiers

The two sources are exactly one month apart, and the split is *total*:

| source | `month` | `day_of_week` | `hour` |
|---|---|---|---|
| `d1` | **2 only** | **3 only** | 3-7 |
| `d2` | **3 only** | 1, 2, 3, 4 | 0-6 |

`month` alone therefore names the source with 100 % accuracy. Fed to a model
alongside the physical inputs it becomes a proxy for "which file is this", and
the model starts answering from the date instead of from the operator's inputs.

The score barely notices -- the behaviour changes completely.


In [5]:
CALENDAR = ["hour", "day_of_week", "month", "is_weekend"]
leaky_features = ENERGY_FEATURES + CALENDAR


def fit_rate_model(features):
    # Fit the (retired) rate formulation on all sources, arbitrary feature list.
    frame = data.combined(datasets)
    x = frame[features].fillna(0)
    y = frame[config.RATE_TARGET]
    mask = y.notna()
    scaler = StandardScaler().fit(x[mask])
    model = config.energy_estimator().fit(scaler.transform(x[mask]), y[mask])
    return model, scaler


def session_kwh(model, scaler, features, current_a, *, duration_h=2.0, month=3, voltage=400.0):
    # Predicted kWh for a duration_h session at `voltage` V / current_a A,
    # 25 C, SoC 50 -- exactly the serving path, scaler included.
    row = {
        "Charging Power_kW": voltage * current_a / 1000.0,  # P = V*I/1000
        "Battery Temperature_C": 25.0,
        "State Of Charge_SoC": 50.0,
        "hour": 4,
        "day_of_week": 2,
        "month": month,
        "is_weekend": 0,
    }
    x = pd.DataFrame([[row[f] for f in features]], columns=features)
    rate = max(float(model.predict(scaler.transform(x))[0]), 0.0)
    return rate * duration_h


retired = fit_rate_model(ENERGY_FEATURES)
leaky = fit_rate_model(leaky_features)

print("--- score (blocked CV R2) ---")
with_cal = evaluate.per_source_report(
    datasets,
    leaky_features,
    config.RATE_TARGET,
    config.energy_estimator,
    metric_target=config.ENERGY_TARGET,
    integrate=evaluate.integrate_energy,
)
without_cal = evaluate.per_source_report(
    datasets,
    ENERGY_FEATURES,
    config.RATE_TARGET,
    config.energy_estimator,
    metric_target=config.ENERGY_TARGET,
    integrate=evaluate.integrate_energy,
)
print(f"  with calendar    {with_cal['weighted_blocked_cv_r2']:+.4f}")
print(f"  without calendar {without_cal['weighted_blocked_cv_r2']:+.4f}")
print(f"  cost of dropping = {with_cal['weighted_blocked_cv_r2'] - without_cal['weighted_blocked_cv_r2']:+.4f}\n")

print("--- behaviour: vary the CURRENT, hold the date fixed ---")
amps = [10, 20, 40, 60, 80]
sweep = pd.DataFrame(
    {
        "current_A": amps,
        "P_kW @400V": [round(400 * a / 1000, 1) for a in amps],
        "retired kWh": [
            round(session_kwh(*retired, ENERGY_FEATURES, a), 2) for a in amps
        ],
        "with-calendar kWh": [
            round(session_kwh(*leaky, leaky_features, a), 2) for a in amps
        ],
    }
)
print(sweep.to_string(index=False))

print("\n--- behaviour: vary the MONTH, hold the current fixed ---")
months = [2, 3]
drift = pd.DataFrame(
    {
        "month": months,
        "retired kWh": [
            round(session_kwh(*retired, ENERGY_FEATURES, 40, month=m), 2) for m in months
        ],
        "with-calendar kWh": [
            round(session_kwh(*leaky, leaky_features, 40, month=m), 2) for m in months
        ],
    }
)
print(drift.to_string(index=False))


--- score (blocked CV R2) ---


  with calendar    +0.9993
  without calendar +0.9977
  cost of dropping = +0.0016

--- behaviour: vary the CURRENT, hold the date fixed ---
 current_A  P_kW @400V  retired kWh  with-calendar kWh
        10         4.0         8.78              54.99
        20         8.0        43.34              54.99
        40        16.0        43.34              54.99
        60        24.0        43.34              55.00
        80        32.0        43.35              55.01

--- behaviour: vary the MONTH, hold the current fixed ---
 month  retired kWh  with-calendar kWh
     2        43.34               8.02
     3        43.34              54.99


### Finding 1b -- the second model, measured

**What the data actually allows.** Three identities hold in the shipped CSVs, and
each one removes a quantity a model could have claimed to learn:

| identity | deviation |
|---|---|
| `Charging Power_kW = V x I / 1000` | **0.055 %** (d1), **0.136 %** (d2) |
| `Charging Rate_kW x Charging_Duration_h = Energy Supplied_kWh` | **exactly 0.00000000 kWh** |
| `Energy Supplied_kWh = integral P dt` | **0.005 kWh** (d1), **0.098 kWh** (d2) |

The second is decisive. The published pipeline predicted `Charging Rate_kW` and
then multiplied it back by `Charging_Duration_h` -- but those two columns *are*
the target, by construction. **The duration cancels.** The +0.9977 therefore
measures exactly one thing: can you predict `Energy / Duration`?

And that quantity is a near-constant, while its only physical driver is not:

| | CV of `Energy/Duration` | CV of `Charging Power_kW` | `corr(rate, power)` |
|---|---|---|---|
| d1 | **2.55 %** | **43.88 %** | **+0.0353** |
| d2 | **3.22 %** | **47.41 %** | **+0.0364** |

The target varies ~17x less than the power that drives it. A 2.6 % signal cannot
carry a 44 % response, and the correlation says it does not.

**What the model did with it.** On the current sweep the formulation steps from
8.78 to 43.34 kWh between 10 A and 20 A and then plateaus -- it answered with
exactly **two values**, 4.392 kW or 21.668 kW, reading a power threshold near
7 kW to decide *"which dataset is this"*. Nothing else was learned.

**And the calendar still hijacks it.** Hold every physical input fixed and change
nothing but `month`: the calendar variant goes from **8.02 kWh to 54.99 kWh -- a
6.9x swing driven entirely by the date.** The no-calendar variant returns 43.34
kWh either way. Dropping calendar costs 0.0016 of blocked-CV R² (0.9993 ->
0.9977) and removes a factor the operator cannot influence.


In [6]:
# --- Finding 1b, part 1: three identities in the shipped CSVs ---------------
# None of these is a modelling claim -- it is arithmetic on the raw columns.
# Each one removes a quantity a model could have claimed to learn.

DURATION = audit_mod.DURATION_COLUMN

for source, frame in datasets.items():
    power = frame["Charging Power_kW"]
    vi = frame["Charging Voltage_V"] * frame["Charging Current_A"] / 1000.0
    energy = frame[config.ENERGY_TARGET]
    rate = frame[config.RATE_TARGET]
    duration = frame[DURATION]

    # integral P dt, with the clock resetting at every session boundary
    elapsed = frame["Time Elapsed_s"].astype(float)
    dt = elapsed.diff()
    boundary = dt.isna() | (dt <= 0)
    step_h = dt.where(~boundary, 0.0).fillna(0.0) / 3600.0
    integral = (power * step_h).groupby(boundary.cumsum().to_numpy()).cumsum()

    pair = frame[[config.RATE_TARGET, "Charging Power_kW"]].dropna()
    clean_rate = rate.replace([np.inf, -np.inf], np.nan).dropna()

    print(f"\n{source}   ({len(frame)} rows, {boundary.cumsum().nunique()} sessions)")
    print(f"  P  == V*I/1000        max relative err  {((power - vi).abs() / vi).max():.4%}")
    print(
        "  rate * duration == E  max absolute err  "
        f"{(rate * duration - energy).abs().max():.8f} kWh"
    )
    print(
        "  integral P dt == E    max absolute err  "
        f"{(energy - integral).abs().max():.6f} kWh"
    )
    print(
        f"  CV(rate) = {clean_rate.std() / clean_rate.mean():.2%}   "
        f"CV(power) = {power.std() / power.mean():.2%}   "
        f"corr(rate, power) = {np.corrcoef(pair.iloc[:, 0], pair.iloc[:, 1])[0, 1]:+.4f}"
    )

print(
    "\nThe middle line is the whole finding: the published pipeline predicts"
    "\n`rate` and multiplies by `duration` again, but `rate * duration` IS the"
    "\ntarget. The duration cancels -- the metric never touches the ramp."
)



d1   (3006 rows, 1 sessions)
  P  == V*I/1000        max relative err  0.0554%
  rate * duration == E  max absolute err  0.00000000 kWh
  integral P dt == E    max absolute err  0.005143 kWh
  CV(rate) = 2.55%   CV(power) = 43.88%   corr(rate, power) = +0.0353

d2   (9087 rows, 4 sessions)
  P  == V*I/1000        max relative err  0.1360%
  rate * duration == E  max absolute err  0.00000000 kWh
  integral P dt == E    max absolute err  0.097786 kWh
  CV(rate) = 3.22%   CV(power) = 47.41%   corr(rate, power) = +0.0364

The middle line is the whole finding: the published pipeline predicts
`rate` and multiplies by `duration` again, but `rate * duration` IS the
target. The duration cancels -- the metric never touches the ramp.


In [7]:
# --- Finding 1b: measure the retired formulation against the reference -------
# The reference is energy = power x duration. It is not an approximation:
# the stored Energy column IS integral P dt (0.005 kWh / 0.098 kWh) and is
# exactly rate x duration. The two inputs V and I already determine power to
# within 0.14 %, so nothing in this sweep is a modelling choice.

pairs = [
    (250, 5), (300, 10), (400, 15), (450, 20), (500, 30),
    (600, 40), (650, 55), (700, 65), (780, 80),
]

print(f"{'V':>5}{'I':>5}{'P kW':>9}{'served kWh':>12}{'P x t kWh':>11}{'error %':>10}")
errors = []
for voltage, amps_in in pairs:
    power = voltage * amps_in / 1000.0
    served = session_kwh(*retired, ENERGY_FEATURES, amps_in, voltage=voltage)
    reference = power * 2.0
    errors.append((served - reference) / reference * 100)
    print(
        f"{voltage:>5}{amps_in:>5}{power:>9.3f}"
        f"{served:>12.3f}{reference:>11.3f}{errors[-1]:>10.1f}"
    )

errors = np.abs(errors)
print(f"\nretired formulation : mean |error| {errors.mean():.1f}%   "
      f"worst {errors.max():.1f}%")
print(f"physics (power x t) : mean |error| 0.0%   worst 0.0%")

constant = np.mean(
    [evaluate.target_is_a_ramp(f)["constant_k_r2"] for f in datasets.values()]
)
print(f"\nconstant_k_r2 = {constant:.4f} -- a CONSTANT fits this target better "
      "than the +0.9977 that was published for it.")


    V    I     P kW  served kWh  P x t kWh   error %
  250    5    1.250       8.783      2.500     251.3
  300   10    3.000       8.783      6.000      46.4
  400   15    6.000       8.783     12.000     -26.8
  450   20    9.000      43.335     18.000     140.8
  500   30   15.000      43.335     30.000      44.5
  600   40   24.000      43.335     48.000      -9.7
  650   55   35.750      43.345     71.500     -39.4
  700   65   45.500      43.345     91.000     -52.4
  780   80   62.400      43.345    124.800     -65.3

retired formulation : mean |error| 75.2%   worst 251.3%
physics (power x t) : mean |error| 0.0%   worst 0.0%

constant_k_r2 = 0.9996 -- a CONSTANT fits this target better than the +0.9977 that was published for it.


### ...and could a better formulation have worked?

Six candidates, one estimator, one protocol, one sweep -- and **two** columns,
because they answer two different questions:

* **blocked CV R²** asks *"can you reconstruct the historical ramp?"*, scoring
  row-wise output against `Energy Supplied_kWh`.
* **served error** asks *"the operator enters V, I and 2 h -- how much energy is
  actually delivered?"*, scored against `P x duration`.

The two rankings are almost reversed. That is the finding: **the published metric
was not merely insensitive, it was the only metric the shipped formulation could
pass**, because its output is algebraically the target. C2 and C3 post a deeply
negative R² for the opposite reason -- they emit a session total, which cannot
match a row-wise cumulative -- while serving within **2.8 %**. Read the right-hand
column for deployment.


In [8]:
# --- Finding 1b: every formulation that could have shipped ----------------
# One estimator for all of them (config.energy_estimator), so the only variable
# is the framing. Two numbers per row, and they disagree:
#
#   blocked CV R2  -- against Energy Supplied_kWh, the quantity actually served
#   served error   -- mean |output - physical P x duration| over a V/I sweep
#
# A formulation can post a great R2 and still be 75 % wrong in service. That
# disagreement is the whole finding: the metric, not the model, was broken.

# derating = Energy / (P x duration) -- the fraction of the possible energy that
# arrived. `rate x duration` is an attempt to predict exactly this ratio.
der_datasets = {
    name: frame.assign(
        derating=frame[config.ENERGY_TARGET]
        / (frame["Charging Power_kW"] * frame[DURATION].replace(0, np.nan))
    )
    for name, frame in datasets.items()
}


def integrate_power(pred, frame):
    """P_hat x duration -- arithmetically the same operation as rate x duration."""
    return evaluate.integrate_energy(pred, frame)


def integrate_derating(pred, frame):
    """derating_hat x P x duration."""
    return (
        np.asarray(pred, dtype=float)
        * frame["Charging Power_kW"].fillna(0).to_numpy()
        * frame[DURATION].fillna(0).to_numpy()
    )


ELEC = ["Battery Temperature_C", "State Of Charge_SoC"]
V_I_ELEC = ["Charging Voltage_V", "Charging Current_A", *ELEC]

CANDIDATES = {
    "C1  [P,T,SoC] -> rate  (shipped, retired)": (
        datasets, ENERGY_FEATURES, config.RATE_TARGET, evaluate.integrate_energy
    ),
    "C2  [P,T,SoC] -> power": (
        datasets, ENERGY_FEATURES, config.POWER_TARGET, integrate_power
    ),
    "C3  [V,I,T,SoC] -> power": (
        datasets, V_I_ELEC, config.POWER_TARGET, integrate_power
    ),
    "C4  [P,T,SoC,duration] -> energy": (
        datasets, [*ENERGY_FEATURES, DURATION], config.ENERGY_TARGET, None
    ),
    "C5  [T,SoC] -> derating": (
        der_datasets, ELEC, "derating", integrate_derating
    ),
}

SWEEP = [
    (250, 5), (300, 10), (400, 15), (450, 20), (500, 30),
    (600, 40), (650, 55), (700, 65), (780, 80),
]


def fit_all(source, features, target):
    """Fit on the pooled rows with a scaler -- the serving path's shape."""
    frame = pd.concat([f.copy() for f in source.values()], ignore_index=True)
    x = frame[features].fillna(0)
    y = frame[target]
    mask = y.notna()
    scaler = StandardScaler().fit(x[mask])
    model = config.energy_estimator().fit(scaler.transform(x[mask]), y[mask])
    return model, scaler


def served_error(source, features, target, integrate_fn):
    model, scaler = fit_all(source, features, target)
    errors = []
    for voltage, amps in SWEEP:
        row = {
            "Charging Voltage_V": voltage,
            "Charging Current_A": amps,
            "Battery Temperature_C": 25.0,
            "State Of Charge_SoC": 50.0,
            "Charging Power_kW": voltage * amps / 1000.0,
            DURATION: 2.0,
            "hour": 4,
            "day_of_week": 2,
            "month": 3,
            "is_weekend": 0,
        }
        frame = pd.DataFrame([row])
        x = frame[features].fillna(0)
        pred = np.asarray([model.predict(scaler.transform(x))[0]])
        out = float(integrate_fn(pred, frame)[0]) if integrate_fn else float(pred[0])
        reference = row["Charging Power_kW"] * row[DURATION]
        errors.append(abs(out - reference) / reference * 100)
    return float(np.mean(errors))


rows = {}
for label, (source, features, target, integrate_fn) in CANDIDATES.items():
    started = time.perf_counter()
    rep = evaluate.per_source_report(
        source,
        features,
        target,
        config.energy_estimator,
        metric_target=config.ENERGY_TARGET,
        integrate=integrate_fn,
    )
    rows[label] = {
        "blocked CV R2": rep["weighted_blocked_cv_r2"],
        "served err %": round(served_error(source, features, target, integrate_fn), 1),
        "eval s": round(time.perf_counter() - started, 1),
    }

rows["C6  no model: energy = P x duration"] = {
    "blocked CV R2": "exact (no fit)",
    "served err %": 0.0,
    "eval s": 0.0,
}
print(pd.DataFrame(rows).T.to_string())

constant = np.mean(
    [evaluate.target_is_a_ramp(f)["constant_k_r2"] for f in datasets.values()]
)
print(
    f"\nFor reference: a CONSTANT fit to the target scores {constant:.4f} -- "
    "better than the +0.9977 published for C1."
)


                                            blocked CV R2 served err % eval s
C1  [P,T,SoC] -> rate  (shipped, retired)          0.9977         74.7   10.9
C2  [P,T,SoC] -> power                            -8.3961          2.8   10.8
C3  [V,I,T,SoC] -> power                          -8.3927          2.8   12.0
C4  [P,T,SoC,duration] -> energy                  -0.0951         82.3   11.1
C5  [T,SoC] -> derating                          -25.2308         36.0   10.2
C6  no model: energy = P x duration        exact (no fit)          0.0    0.0

For reference: a CONSTANT fit to the target scores 0.9996 -- better than the +0.9977 published for C1.


## 1.4 The cheapest leak detector: univariate R²

Fit **one** feature alone, in-sample, and look at the score. A single column
reaching ~1.0 is carrying the answer -- no real physical relationship is that
one-sided across a whole dataset.

In [9]:
print("=" * 74)
print("UNIVARIATE R2 -- one feature fitted alone (in-sample)")
print("=" * 74)
for source, frame in datasets.items():
    print(f"\n{source}   target = {config.ENERGY_TARGET}")
    for column, value in audit_mod.univariate_r2(
        frame, config.ENERGY_TARGET, top=6
    ).items():
        flag = "   <-- LEAK" if value > 0.9 else ""
        print(f"    {column:30} {value:.4f}{flag}")

print("\n" + "=" * 74)
print("COLUMNS EXCLUDED FROM EVERY FEATURE SET, AND WHY")
print("=" * 74)
for column, reason in audit_mod.EXCLUSION_REASONS.items():
    print(f"    {column:26} {reason}")

print(
    f"\nRetained despite correlation: {audit_mod.RETAINED_DESPITE_CORRELATION}"
    "\n    It correlates +1.0 with the cumulative target inside ev_charging_data1.csv,"
    "\n    but NOT with the rate that formulation predicts -- so the"
    "\n    measurement (see soc_ablation in leakage_audit.json), not the correlation,"
    "\n    decides it."
)


UNIVARIATE R2 -- one feature fitted alone (in-sample)

d1   target = Energy Supplied_kWh
    State Of Charge_SoC            1.0000   <-- LEAK
    Time Elapsed_s                 1.0000   <-- LEAK
    Charging_Duration_h            1.0000   <-- LEAK
    hour                           0.9538   <-- LEAK
    Charging Rate_kW               0.5347
    Charging Voltage_V             0.0677

d2   target = Energy Supplied_kWh


    Time Elapsed_s                 0.9994   <-- LEAK
    Charging_Duration_h            0.9994   <-- LEAK
    hour                           0.9754   <-- LEAK
    State Of Charge_SoC            0.9166   <-- LEAK
    Charging Rate_kW               0.4316
    Battery Voltage_BV             0.0388

COLUMNS EXCLUDED FROM EVERY FEATURE SET, AND WHY
    Time Elapsed_s             cumulative elapsed time -- the ramp the target is built from
    Charging_Duration_h        same ramp; reapplied explicitly as energy = rate * duration
    Energy Supplied_kWh        the cumulative target itself
    Charging Rate_kW           the energy model's training target
    Charging Power_kW          the power model's training target (an input to the energy model)
    Time-lap                   constant (always 10 s) -- carries no information
    Regenerative Power_kW      all zeros in ev_charging_data1.csv
    Battery Voltage_BV         near-duplicate of Charging Voltage_V; ev_charging_data2 only
    Battery

# 2. Feature contracts

`ml/pipeline/config.py` is the single source of truth. The notebook imports the
same lists the Django view builds its input frame from, and `meta.json` records
them so the loader can assert the order at load time -- a retrained model
cannot silently be fed columns in a different order.

In [10]:
contract = {
    name: {
        "algorithm": spec["algorithm"],
        "target": spec["target"],
        "features": ", ".join(spec["features"]),
        "derived": spec["derived"] or "-",
        "view reads": spec["output"],
    }
    for name, spec in SPECS.items()
}
pd.DataFrame(contract).T

,algorithm,target,features,derived,view reads
power,XGBRegressor,Charging Power_kW,"Charging Voltage_V, Charging Current_A, Batter...",energy_kwh = charging_power_kw * duration_h,charging_power_kw


# 3. Honest evaluation

Two protocols, both reported, because they answer different questions.

* **Chronological 80/20 holdout** -- train on the past, test on the future
  within one source. This is the deployment question: can the model predict the
  *next* chunk of time? A random split is unusable here (it inflates R² from
  0.95 to 1.00 on near-duplicate rows).
* **Blocked `TimeSeriesSplit(5)`** -- folds never span two sources, so no fold
  trains on 1-7 kW data and tests on 5-50 kW data.

The weighted mean is the headline; the per-source rows are what make it
auditable, because they show *where* the model generalises.

In [11]:
power_report = evaluate.per_source_report(
    datasets,
    POWER_FEATURES,
    SPECS[POWER_MODEL_NAME]["target"],
    SPECS[POWER_MODEL_NAME]["estimator"],
    scaler_factory=StandardScaler,
)


def flatten(label, rep):
    row = {"blocked CV R2 (weighted)": rep["weighted_blocked_cv_r2"]}
    row.update({f"chrono {s}": v["r2"] for s, v in rep["chronological_holdout"].items()})
    row.update({f"blocked {s}": v["r2"] for s, v in rep["blocked_cv"].items()})
    return {label: row}


pd.DataFrame(flatten("power (XGB)", power_report)).T.round(4)


,blocked CV R2 (weighted),chrono d1,chrono d2,blocked d1,blocked d2
power (XGB),0.996,0.996,0.9984,0.9917,0.9974


In [12]:
# Why the sources must not be pooled into one CV run.
pooled = evaluate.evaluate_blocked(
    data.combined(datasets),
    POWER_FEATURES,
    config.POWER_TARGET,
    SPECS[POWER_MODEL_NAME]["estimator"],
    scaler_factory=StandardScaler,
)

print(f"pooled TimeSeriesSplit across both sources : R2 = {pooled.r2:+.4f}")
print(
    f"per-source blocked CV (row-weighted)       : "
    f"R2 = {power_report['weighted_blocked_cv_r2']:+.4f}"
)
print()
print("The pooled number is dragged down by the single fold that straddles the")
print("d1 -> d2 boundary: it trains on 1-7 kW / 200-500 V data and is tested on")
print("5-50 kW / 300-800 V data. Reporting one pooled figure would understate a")
print("model that scores 0.996 inside each regime.")

pooled TimeSeriesSplit across both sources : R2 = +0.7086
per-source blocked CV (row-weighted)       : R2 = +0.9960

The pooled number is dragged down by the single fold that straddles the
d1 -> d2 boundary: it trains on 1-7 kW / 200-500 V data and is tested on
5-50 kW / 300-800 V data. Reporting one pooled figure would understate a
model that scores 0.996 inside each regime.


# 4. Model bake-off

The shipped `power` model is XGBoost rather than RandomForest. That choice was
made on **latency and size**, not accuracy -- the numbers below show why it is
the right trade.

In [13]:
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor

candidates = {
    "RandomForest": lambda: RandomForestRegressor(
        n_estimators=100, random_state=42, n_jobs=-1
    ),
    "HistGradientBoosting": lambda: HistGradientBoostingRegressor(random_state=42),
    "XGBoost (shipped)": config.power_estimator,
}

rows = {}
for label, factory in candidates.items():
    started = time.perf_counter()
    rep = evaluate.per_source_report(
        datasets,
        POWER_FEATURES,
        config.POWER_TARGET,
        factory,
        scaler_factory=StandardScaler,
    )
    rows[label] = {
        "blocked CV R2": rep["weighted_blocked_cv_r2"],
        "chrono d1": rep["chronological_holdout"]["d1"]["r2"],
        "chrono d2": rep["chronological_holdout"]["d2"]["r2"],
        "eval s": round(time.perf_counter() - started, 1),
    }

pd.DataFrame(rows).T.round(4)

,blocked CV R2,chrono d1,chrono d2,eval s
RandomForest,0.9986,0.9990,0.9997,3.9
HistGradientBoosting,0.9983,0.9983,0.9993,2.6
XGBoost (shipped),0.9960,0.9960,0.9984,4.4


In [14]:
import io

frame = data.combined(datasets)
x_all = frame[POWER_FEATURES].fillna(0)
y_all = frame[config.POWER_TARGET]
scaler_all = StandardScaler().fit(x_all)
sample = scaler_all.transform(x_all.iloc[:1])


def cost_of(factory):
    import joblib

    model = factory().fit(scaler_all.transform(x_all), y_all)
    model.predict(sample)  # warm up
    started = time.perf_counter()
    for _ in range(50):
        model.predict(sample)
    ms = (time.perf_counter() - started) / 50 * 1000
    buffer = io.BytesIO()
    joblib.dump(model, buffer)
    return round(ms, 3), round(buffer.tell() / 1e6, 2)


cost = pd.DataFrame(
    {
        label: dict(zip(("ms/row", "MB"), cost_of(factory), strict=True))
        for label, factory in candidates.items()
    }
).T

print("Per-row serving cost (the number that decides the shipping choice):")
print(cost.to_string())

Per-row serving cost (the number that decides the shipping choice):
                      ms/row      MB
RandomForest          32.025  109.78
HistGradientBoosting   1.185    0.37
XGBoost (shipped)      0.330    3.20


RandomForest buys 0.0026 of blocked-CV R² for **~30 ms/row and 109.78 MB**
versus XGBoost's 0.35 ms/row and 3.20 MB -- a ~90× latency penalty and a 34×
size penalty on a model that serves one row per form submission. On accuracy
this is a tie (0.9986 vs 0.9960); on the serving budget it is not, and it is
the serving budget that fails.

# 5. Train and export

`train_all` runs the audit first (so its filename can be recorded in every
`meta.json`), fits the model, evaluates it under both protocols, and writes
versioned artifacts:

```
ml/artifacts/
    current.json                 {"power": "power-v1"}
    model.joblib, scaler.joblib   legacy alias == the active power model
    power-v1/   {model,scaler}.joblib + meta.json
    leakage_audit.json
```

`_write_current` **prunes any pointer whose version directory is gone**, so
retiring a model cannot leave `current.json` naming a directory git never
shipped. Training also prints a shipping note if a freshly created version is
not git-tracked.

Each `meta.json` carries the feature order, the input ranges the serving layer
validates against, both evaluation protocols, latency, size, `derived` (the
arithmetic that turns this output into kWh), the training environment and the
protocol string -- everything needed to audit the artifact without the notebook.


In [15]:
# `_next_version` counts existing `{name}-v*` dirs, so without this a second run
# exports power-v2, power-v3, ... and leaves `current.json` pointing at a
# directory `.gitignore` does not ship. The artifact is regenerable from the
# CSVs committed in data/raw/, so clear the previous export first: every run of
# this notebook then reproduces the same repository state.
for stale in sorted(train.export_root().glob("power-v*")):
    shutil.rmtree(stale)
    print("cleared", stale.name)

results = train.train_all(datasets)
print(train.summarise(results))


cleared power-v1


power (XGBRegressor) -> power-v1
    target      : Charging Power_kW
    features    : Charging Voltage_V, Charging Current_A, Battery Temperature_C, State Of Charge_SoC, hour, day_of_week, month
    derived     : energy_kwh = charging_power_kw * duration_h
    blocked CV  : +0.9960  [d1=+0.9917, d2=+0.9974]
    chrono 80/20: d1=+0.9960, d2=+0.9984
    cost        : 0.329 ms/row, 3.20 MB
    trained_at  : 2026-10-05T17:35:59+00:00



# 6. Serving contract

Reproduce, from `meta.json` alone, exactly what `apps/prediction/views.py`
does. If this cell and the Django view ever disagree, the contract is broken --
`meta.json` is what both read.

In [16]:
import joblib

root = pathlib.Path(settings.MODEL_DIR)
pointers = json.loads((root / "current.json").read_text(encoding="utf-8"))
print("current.json:", pointers)


def predict_session(**form):
    # meta.json -> ordered frame -> power, then arithmetic for kWh.
    values = {
        "Charging Voltage_V": form["voltage"],
        "Charging Current_A": form["current"],
        "Battery Temperature_C": form["battery_temp"],
        "State Of Charge_SoC": form["soc"],
        "Charging_Duration_h": form["duration"],
        **data.calendar_values(pd.Timestamp(form["timestamp"]).to_pydatetime()),
    }

    pdir = root / pointers["power"]
    pmeta = json.loads((pdir / "meta.json").read_text(encoding="utf-8"))
    pmodel = joblib.load(pdir / "model.joblib")
    pscaler = joblib.load(pdir / "scaler.joblib")
    pframe = pd.DataFrame(
        [[values[f] for f in pmeta["features"]]], columns=pmeta["features"]
    )
    power = max(float(pmodel.predict(pscaler.transform(pframe))[0]), 0.0)

    # No second artifact: energy is the rate integrated over the duration.
    rate = power
    energy = rate * form["duration"]

    return {
        "power_kW": round(power, 3),
        "rate_kW": round(rate, 3),
        "energy_kWh": round(energy, 3),
        "I_from_P_V (A)": round(power * 1000.0 / form["voltage"], 2),
    }


for form in (
    dict(
        voltage=400, current=15, battery_temp=25, soc=50,
        duration=2, timestamp="2026-10-01T14:00",
    ),
    dict(
        voltage=600, current=80, battery_temp=32, soc=50,
        duration=4, timestamp="2026-10-01T14:00",
    ),
):
    print(f"{form}  ->  {predict_session(**form)}")


current.json: {'power': 'power-v1'}
{'voltage': 400, 'current': 15, 'battery_temp': 25, 'soc': 50, 'duration': 2, 'timestamp': '2026-10-01T14:00'}  ->  {'power_kW': 6.729, 'rate_kW': 6.729, 'energy_kWh': 13.459, 'I_from_P_V (A)': 16.82}
{'voltage': 600, 'current': 80, 'battery_temp': 32, 'soc': 50, 'duration': 4, 'timestamp': '2026-10-01T14:00'}  ->  {'power_kW': 47.495, 'rate_kW': 47.495, 'energy_kWh': 189.98, 'I_from_P_V (A)': 79.16}


In [17]:
# The contract itself: artifact feature order must equal config.py feature order.
for name, pointer in pointers.items():
    meta = json.loads((root / pointer / "meta.json").read_text(encoding="utf-8"))
    assert meta["features"] == SPECS[name]["features"], (
        f"{name}: artifact {meta['features']} != config {SPECS[name]['features']}"
    )
    assert meta["derived"] == config.ENERGY_DERIVED, meta["derived"]
    print(
        f"{name:7} {pointer:12} {len(meta['features'])} features  OK   "
        f"blocked CV R2 = {meta['evaluation']['weighted_blocked_cv_r2']:+.4f}  "
        f"{meta['latency_ms_1row']:.3f} ms/row  "
        f"{meta['size_bytes_model'] / 1e6:.2f} MB"
    )
print("\nfeature order matches ml/pipeline/config.py for every shipped model.")
print(f"derived matches config.ENERGY_DERIVED: {config.ENERGY_DERIVED}")


power   power-v1     7 features  OK   blocked CV R2 = +0.9960  0.329 ms/row  3.20 MB

feature order matches ml/pipeline/config.py for every shipped model.
derived matches config.ENERGY_DERIVED: energy_kwh = charging_power_kw * duration_h


# Conclusions

**The number that matters.** Blocked CV **R² = +0.9960** for `power`, reached
with a chronological protocol that never lets a fold train on the future or
across the two operating regimes.

**What made the difference** -- in order of size:

1. **Delete the second model, then change the target.** The cumulative
   `Energy Supplied_kWh` is a ramp (`corr(E, Duration) = +1.0000`) that trees
   cannot extrapolate: **−3.16** chronologically, **−3.26** under blocked CV. The natural fix --
   predict the stationary *rate* and integrate -- scored **+0.9977**. That
   score is algebraically circular: `rate x duration` *is* `Energy` in this
   data, so the metric only ever asked "can you predict a near-constant".
   In service it served **75 % mean error** against `power x duration`,
   answered with exactly two values (4.392 or 21.668 kW), and a *constant*
   beat it (0.9999 / 0.9994). Nothing was left to learn, so there is no second
   model: `energy = power x duration`, exact by construction.
2. **Split by time, not at random.** A random split on a 5 s series leaks the
   next near-duplicate row into training and inflates R² from 0.95 to 1.00.
3. **Block the CV per source.** Pooling the two regimes into one `TimeSeriesSplit`
   scores **0.7086** on a model that scores 0.996 inside each of them -- the
   single straddling fold trains on 1-7 kW and tests on 5-50 kW.
4. **Drop the calendar features from any target a date can name.** `month` names
   the source perfectly, so with them the answer is driven by the date:
   **8.02 kWh vs 54.99 kWh for identical physical inputs**. They are *kept* for
   `power`, where they disambiguate the low- and high-power sources at a shared
   (V, I) point (cost of dropping: 7.2 kW of error at 600 V / 80 A).
5. **Keep duration out of the inputs.** It is part of the target; the view
   reapplies it explicitly as `energy = power x duration`.

**Model choice.** XGBoost for `power` (**0.34 ms/row, 3.20 MB**) over
RandomForest (~30 ms/row, 109.78 MB) for 0.0026 of R² -- and over the
RandomForest it originally replaced (9.0 ms/row, 21.9 MB) at equivalent
accuracy. Dropping the second model also halved inference: one artifact load and
one predict per request, and the kWh figure can no longer disagree with the
power figure beside it.

**What would be needed to learn energy instead:** a dataset whose sessions
actually vary in power -- this one has five sessions, two regimes, and a
stationary power series in each.

**Reproduce with:**

```bash
python scripts/train_models.py            # audit + model + report
python scripts/train_models.py --report   # print the audit only
```
